--------------------------------------------------------------------

#First Delivery

--------------------------------------------------------------------

## Dataset Source and Justification

The dataset used in this project is the **Synthetic Financial Datasets For Fraud Detection** from Kaggle.
It contains simulated transactions with labels indicating whether each is fraudulent (`is_fraud = 1`) or not (`is_fraud = 0`).

This dataset was chosen because:
- It closely resembles real-world credit card transaction behavior.
- It contains realistic challenges such as class imbalance, time-dependence, and categorical complexity.
- It is publicly available, well-documented, and commonly used for benchmarking fraud detection systems.

### Target Variable Definition

The target variable for this classification problem is `is_fraud`, a binary variable:
- `0` → Legitimate transaction  
- `1` → Fraudulent transaction  

The goal is to identify fraudulent transactions based on transaction metadata such as amount, category, location, and timing.


###Variable Library
index - Unique Identifier for each row

trans_date_trans_time - Transaction DateTime

cc_num - Credit Card Number of Customer

merchant - Merchant Name

category - Category of Merchant

amt - Amount of Transaction

first - First Name of Credit Card Holder

last - Last Name of Credit Card Holder

gender - Gender of Credit Card Holder

street - Street Address of Credit Card Holder

city - City of Credit Card Holder

state - State of Credit Card Holder

zip - Zip of Credit Card Holder

lat - Latitude Location of Credit Card Holder

long - Longitude Location of Credit Card Holder

city_pop - Credit Card Holder's City Population

job - Job of Credit Card Holder

dob - Date of Birth of Credit Card Holder

trans_num - Transaction Number

unix_time - UNIX Time of transaction

merch_lat - Latitude Location of Merchant

merch_long - Longitude Location of Merchant

is_fraud - Fraud Flag <--- Target Class



In [ ]:
# Third-party libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from datetime import datetime
from google.colab import drive
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import LabelEncoder

# ML Models
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# ML Stats
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)



print("Libraries imported successfully.")

In [ ]:
# Upload kaggle .json file here
from google.colab import files
files.upload()


In [ ]:
os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json


In [ ]:
DATA_DIR = "data"
DATASET = "kartik2112/fraud-detection"

os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d {DATASET} -p {DATA_DIR}
!unzip -o {DATA_DIR}/*.zip -d {DATA_DIR}


In [ ]:
bank_df_train = pd.read_csv("data/fraudTrain.csv")
bank_df_test  = pd.read_csv("data/fraudTest.csv")

print(bank_df_train.shape, bank_df_test.shape)
bank_df_train.head()


In [ ]:
fraud_ratio = (bank_df_train['is_fraud'].mean()) * 100
print(f"Percentage of Fraudulent Transactions: {fraud_ratio:.2f}%")


In [ ]:
bank_df_train['cc_num'].drop_duplicates().shape[0]

In [ ]:
bank_df_train[['cc_num', 'first', 'last']].drop_duplicates().shape[0]

In [ ]:
bank_df_train.head(3)

In [ ]:
bank_df_test.head(3)

In [ ]:
bank_df_train.info()

In [ ]:
bank_df_test.info()


--------------------------------------------------------------------
## Feature Engineering
--------------------------------------------------------------------


In [ ]:
print("Missing values per column:")
print(bank_df_train.isnull().sum())

In [ ]:
bank_df_train.describe().round(2)

In [ ]:
# Remove duplicate rows if any
duplicates = bank_df_train.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    bank_df_train.drop_duplicates(inplace=True)
    print("Duplicates dropped.")
else:
    print("No duplicates found.")

In [ ]:
for_training = bank_df_train.copy()
for_training.head(3)

In [ ]:
# Checking if all transaction numbers are unique
for_training['trans_num'].nunique()


In [ ]:
# Drop all unnecessary columns and transaction number since all are unique and will not provide any statistical value
cols_to_keep = ['trans_date_trans_time', 'cc_num', 'category',
       'amt', 'gender', 'city', 'state', 'zip',
       'city_pop', 'job', 'dob',
       'merch_lat', 'merch_long', 'is_fraud']

for_training = for_training[cols_to_keep]
for_training.head(3)

In [ ]:
# Derive customer age from Date of Birth (dob)
for_training['dob'] = pd.to_datetime(for_training['dob'], errors='coerce')
for_training['Age'] = datetime.now().year - for_training['dob'].dt.year
for_training.head(3)

In [ ]:
try:
  for_training.drop(columns=['dob'], inplace=True)
  for_training.head()
except:
  for_training.head()

for_training.head()

In [ ]:
for_training['trans_date_trans_time'] = pd.to_datetime(for_training['trans_date_trans_time'])
for_training.info()

In [ ]:
for_training = for_training.sort_values(['cc_num', 'trans_date_trans_time'])

for_training['time_since_last_trans'] = (
    for_training.groupby('cc_num')['trans_date_trans_time']
    .diff()
)

for_training['time_since_last_trans_sec'] = for_training['time_since_last_trans'].dt.total_seconds()
for_training['time_since_last_trans_min'] = for_training['time_since_last_trans_sec'] / 60
for_training['time_since_last_trans_hour'] = for_training['time_since_last_trans_sec'] / 3600

for_training.head(3)

In [ ]:
# To avoid problems we replace the NaN with
nan_cols = ['time_since_last_trans_sec',
       'time_since_last_trans_min', 'time_since_last_trans_hour']

for_training[nan_cols] = for_training[nan_cols].fillna(0)

for_training.info()




In [ ]:
# Compute the average transaction amount per customer (cc_num)
for_training['avg_trans_amt'] = for_training.groupby('cc_num')['amt'].transform('mean')

for_training['amt_ratio'] = for_training['amt'] / for_training['avg_trans_amt']

for_training['amt_ratio'] = for_training['amt_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0)

# Check new column
for_training[['cc_num', 'amt', 'avg_trans_amt', 'amt_ratio']].head()


In [ ]:
# --- Define job-to-sector mapping ---
job_sector_map = {
    # --- Engineering & Technical ---
    'Engineer': 'Engineering & Technical',
    'Engineering': 'Engineering & Technical',
    'Architect': 'Engineering & Technical',
    'Technologist': 'Engineering & Technical',
    'Surveyor': 'Engineering & Technical',
    'Scientist': 'Science & Research',
    'Geologist': 'Science & Research',
    'Hydrologist': 'Science & Research',
    'Geophysicist': 'Science & Research',
    'Seismologist': 'Science & Research',
    'Analyst': 'Science & Research',
    'Developer': 'Information Technology',
    'Programmer': 'Information Technology',
    'IT': 'Information Technology',
    'Systems': 'Information Technology',
    'Database': 'Information Technology',
    'Network': 'Information Technology',
    'Data': 'Information Technology',

    # --- Education & Training ---
    'Teacher': 'Education',
    'Lecturer': 'Education',
    'Professor': 'Education',
    'Trainer': 'Education',
    'Education': 'Education',
    'Tutor': 'Education',

    # --- Healthcare & Life Sciences ---
    'Doctor': 'Healthcare',
    'Nurse': 'Healthcare',
    'Surgeon': 'Healthcare',
    'Therapist': 'Healthcare',
    'Pharmacist': 'Healthcare',
    'Psychologist': 'Healthcare',
    'Pathologist': 'Healthcare',
    'Paramedic': 'Healthcare',
    'Physiotherapist': 'Healthcare',
    'Dietitian': 'Healthcare',
    'Optician': 'Healthcare',
    'Dentist': 'Healthcare',

    # --- Business, Finance & Management ---
    'Accountant': 'Finance & Business',
    'Banker': 'Finance & Business',
    'Trader': 'Finance & Business',
    'Consultant': 'Finance & Business',
    'Adviser': 'Finance & Business',
    'Manager': 'Finance & Business',
    'Executive': 'Finance & Business',
    'Officer': 'Finance & Business',
    'Coordinator': 'Finance & Business',
    'Administrator': 'Finance & Business',
    'Strategist': 'Finance & Business',
    'Chief': 'Finance & Business',

    # --- Legal & Public Administration ---
    'Lawyer': 'Legal & Public Service',
    'Solicitor': 'Legal & Public Service',
    'Barrister': 'Legal & Public Service',
    'Judge': 'Legal & Public Service',
    'Attorney': 'Legal & Public Service',
    'Officer': 'Legal & Public Service',
    'Civil Service': 'Legal & Public Service',
    'Inspector': 'Legal & Public Service',
    'Police': 'Legal & Public Service',
    'Prison': 'Legal & Public Service',

    # --- Creative Arts, Media & Design ---
    'Artist': 'Arts & Media',
    'Designer': 'Arts & Media',
    'Journalist': 'Arts & Media',
    'Editor': 'Arts & Media',
    'Producer': 'Arts & Media',
    'Photographer': 'Arts & Media',
    'Musician': 'Arts & Media',
    'Actor': 'Arts & Media',
    'Dancer': 'Arts & Media',
    'Curator': 'Arts & Media',
    'Writer': 'Arts & Media',

    # --- Service, Retail & Hospitality ---
    'Sales': 'Service & Retail',
    'Retail': 'Service & Retail',
    'Buyer': 'Service & Retail',
    'Barista': 'Service & Retail',
    'Restaurant': 'Service & Retail',
    'Hospitality': 'Service & Retail',
    'Hotel': 'Service & Retail',
    'Tour': 'Service & Retail',
    'Travel': 'Service & Retail',
    'Catering': 'Service & Retail',

    # --- Agriculture & Environment ---
    'Agricultural': 'Agriculture & Environment',
    'Horticulturist': 'Agriculture & Environment',
    'Forestry': 'Agriculture & Environment',
    'Environment': 'Agriculture & Environment',
    'Ecologist': 'Agriculture & Environment',
    'Conservation': 'Agriculture & Environment',
    'Farm': 'Agriculture & Environment',

    # --- Public Service & Social Work ---
    'Social': 'Public Service',
    'Counsellor': 'Public Service',
    'Therapist': 'Public Service',
    'Psychotherapist': 'Public Service',
    'Advisor': 'Public Service',
    'Officer': 'Public Service',
    'Development': 'Public Service',
    'Worker': 'Public Service',
    'Aid': 'Public Service',
    'Volunteer': 'Public Service',

    # --- Transport, Logistics & Infrastructure ---
    'Pilot': 'Transport & Logistics',
    'Air': 'Transport & Logistics',
    'Logistics': 'Transport & Logistics',
    'Driver': 'Transport & Logistics',
    'Planner': 'Transport & Logistics',
    'Transport': 'Transport & Logistics',
    'Shipping': 'Transport & Logistics',
    'Engineer, drilling': 'Transport & Logistics',

    # --- Miscellaneous ---
    'Other': 'Other / Miscellaneous'
}

# --- Apply mapping ---
def map_job_to_sector(job_title):
    for keyword, sector in job_sector_map.items():
        if keyword.lower() in job_title.lower():
            return sector
    return 'Other / Miscellaneous'

for_training['job_sector'] = for_training['job'].apply(map_job_to_sector)

for_training.head()


### Data Cleaning and Feature Engineering

1. **Missing Values:** The dataset contains no missing entries (verified using `df.isnull().sum()`).

2. **Datetime Conversion:** The `trans_date_trans_time` column was converted to datetime format to enable temporal analysis.

3. **Age Calculation:** The `dob` column was used to derive each client’s age in years.

4. **Time Since Last Transaction:** Computed per card (`cc_num`) to measure the time difference between consecutive transactions and capture customer spending frequency.

5. **Amount Ratio:** Calculated the average transaction amount per card (`cc_num`) and derived the ratio `amt_ratio = amt / avg_trans_amt` to identify transactions that deviate from a customer’s usual spending behavior.

6. **Job Categorization:** Grouped detailed job titles into broader occupational sectors to simplify visualization and improve interpretability.


##Data Visualization

In [ ]:
for_training['hour'] = pd.to_datetime(for_training['trans_date_trans_time']).dt.hour


cols_to_keep = ['amt', 'city_pop', 'Age', 'is_fraud', 'time_since_last_trans','hour','amt_ratio' ]
bank_numeric = for_training[cols_to_keep]
corr = bank_numeric.corr()

plt.figure(figsize=(8, 6))  # adjust size as needed
sns.heatmap(
    corr,
    annot=True,       # show correlation values
    cmap='coolwarm',  # color scheme
    fmt='.2f',        # number format
    linewidths=0.5,   # grid lines
    square=True       # make cells square
)
plt.title('Correlation Heatmap of Bank Features', fontsize=14)
plt.show()





In [ ]:
# Select only numeric columns
numeric_cols = for_training.select_dtypes(include=['number'])

# Compute correlations with is_fraud
fraud_corr = numeric_cols.corr()['is_fraud'].sort_values(ascending=False)

# Drop is_fraud itself (since it will be 1.0)
fraud_corr = fraud_corr.drop('is_fraud')

# Plot
plt.figure(figsize=(8, 5))
sns.barplot(x=fraud_corr.values, y=fraud_corr.index, palette='coolwarm')

plt.title('Correlation of Features with Fraudulent Transactions (is_fraud)', fontsize=14)
plt.xlabel('Correlation Coefficient')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:

num_cols = [
    'amt',
    'city_pop',
    'Age',
    'time_since_last_trans_hour',
    'is_fraud',
    'hour',
    'amt_ratio'
]

sample_df = for_training[num_cols].sample(10000, random_state=42)

# Create pairplot
sns.pairplot(
    data=sample_df,
    vars=['amt', 'city_pop', 'Age', 'time_since_last_trans_hour'],
    hue='is_fraud',
    diag_kind='kde',
    corner=True,
    plot_kws={'alpha': 0.5, 's': 15}
)

plt.suptitle('Pairwise Relationships Among Numerical Features (Sample of 10,000)', y=1.02, fontsize=14)
plt.show()


In [ ]:
bank_numeric.describe().round(2)


--------------------------------------------------------------------
## Customer Analysis
--------------------------------------------------------------------



In [ ]:
# Graphical Visualization of Dataset
sns.set(style="whitegrid")

# Set axes
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Transaction Amount (Linear)
sns.histplot(for_training['amt'], kde=True, bins=100, ax=axes[0, 0])
axes[0, 0].set_title('Transaction Amount (Linear Scale)')
axes[0, 0].set_xlabel('amt')

# Transaction Amount (Log)
sns.histplot(for_training['amt'], kde=True, bins=100, ax=axes[0, 1])
axes[0, 1].set_xscale('log')
axes[0, 1].set_title('Transaction Amount (Log Scale)')
axes[0, 1].set_xlabel('amt (log scale)')

# Age
sns.histplot(for_training['Age'], kde=True, bins=20, ax=axes[1, 0], color='darkolivegreen')
axes[1, 0].set_title('Age Distribution')
axes[1, 0].set_xlabel('Age')

# Time Since Last Transaction (Hours)
sns.histplot(for_training['time_since_last_trans_hour'], kde=True, bins=100, ax=axes[1, 1], color='darkolivegreen')
axes[1, 1].set_title('Time Since Last Transaction (Hours)')
axes[1, 1].set_xlabel('Hours since last transaction')

# Gender
sns.countplot(data=for_training, x='gender', ax=axes[2, 0], palette='pastel')
axes[2, 0].set_title('Gender Distribution Among Clients')
axes[2, 0].set_xlabel('Gender')


sns.countplot(data=for_training, x='hour', ax=axes[2, 1], color='#FFB347')
axes[2, 1].set_title('Number of Transactions per Hour of the Day')
axes[2, 1].set_xlabel('Hour of Day (0–23)')
axes[2, 1].set_ylabel('Transaction Count')



plt.tight_layout()
plt.show()


###Analysis of Customer Base

The exploratory visualizations reveal several key behavioral patterns among customers.
Most transactions are relatively small, typically between \$0 and $100, as shown in both linear and log-scale plots.
The customer base is concentrated in the 40–50 age range, reflecting a predominantly middle-aged demographic.
Female clients account for the majority of transactions in the dataset.
Transaction frequency increases sharply around midday and remains high until late evening (12:00–23:00), aligning with typical consumer activity hours.
Additionally, the “time since last transaction” distribution shows that many customers make consecutive purchases within short intervals, suggesting habitual or clustered spending patterns.

In [ ]:
# This explains the strange left-skewed graph for transaction amount
max_amt = for_training['amt'].max()
bin_edges = np.arange(0, max_amt + 100, 100)

for_training['amt_bin'] = pd.cut(for_training['amt'], bins=bin_edges, right=False)

amt_distribution = for_training['amt_bin'].value_counts().sort_index()
print(amt_distribution)

In [ ]:
for_training['job_sector'].value_counts()

--------------------------------------------------------------------
## Fraud Analysis
--------------------------------------------------------------------



In [ ]:
top_10_states = for_training['state'].value_counts().nlargest(10).index

ax = sns.countplot(
    data=for_training[for_training['state'].isin(top_10_states)],
    y='state',
    order=top_10_states,
    hue='is_fraud'
)
plt.title('State Distribution of Fraudulent Transactions')
plt.xlabel('Count')
plt.ylabel('State')


ax.yaxis.label.set_size(10)

plt.tight_layout()
plt.show()

We can observe here that the customer distribution of this bank in particular is mainly consentrated in the bigger states like Texas and New York likely due to the concentration of the population. The amount of fraud we see seems to be **roughly proportional (~0.98)** to the state it occurs in due to the.

*Note:* The amount of unique jobs and cities listed in the data set make it difficult to graph it correctly. Therefor we may have to resort to grouping jobs by category

In [ ]:
for_training['is_fraud'] = for_training['is_fraud'].astype(int)

state_summary = (
    for_training.groupby('state')
    .agg(
        total_transactions=('is_fraud', 'count'),
        fraud_transactions=('is_fraud', 'sum'),
        avg_city_pop=('city_pop', 'mean')
    )
    .reset_index()
)

top10_states = (
    state_summary.sort_values('avg_city_pop', ascending=False)
    .head(10)
)

top10_states['fraud_rate_%'] = (
    top10_states['fraud_transactions'] / top10_states['total_transactions'] * 100
)

top10_states = top10_states.sort_values('total_transactions', ascending=False)

print("Top 10 Most Populated States Summary:\n")
print(top10_states[['state', 'avg_city_pop', 'total_transactions', 'fraud_transactions', 'fraud_rate_%']])

corr = top10_states['total_transactions'].corr(top10_states['fraud_transactions'])
print(f"\nCorrelation between total transactions and fraud counts: {corr:.2f}")


In [ ]:
fraud_df = for_training[for_training['is_fraud'] == 1]
fraud_df.head()

In [ ]:
# Graphical Visualization of Fraud
fraud_df['trans_date_trans_time'] = pd.to_datetime(fraud_df['trans_date_trans_time'])
fraud_df['time_since_last_trans_hour'] = fraud_df['time_since_last_trans'].dt.total_seconds() / 3600

# Set the axes
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Transaction Amount (Linear)
sns.histplot(fraud_df['amt'], kde=True, bins=100, ax=axes[0, 0])
axes[0, 0].set_title('Transaction Amount (Linear Scale)')
axes[0, 0].set_xlabel('amt')

# Transaction Amount (Log)
sns.histplot(fraud_df['amt'], kde=True, bins=100, ax=axes[0, 1])
axes[0, 1].set_xscale('log')
axes[0, 1].set_title('Transaction Amount (Log Scale)')
axes[0, 1].set_xlabel('amt (log scale)')

# Age
sns.histplot(fraud_df['Age'], kde=True, bins=20, ax=axes[1, 0], color='darkolivegreen')
axes[1, 0].set_title('Age Distribution')
axes[1, 0].set_xlabel('Age')

# Time Since Last Transaction (Hours)
sns.histplot(fraud_df['time_since_last_trans_hour'], kde=True, bins=100, ax=axes[1, 1], color='darkolivegreen')
axes[1, 1].set_title('Time Since Last Transaction (Hours)')
axes[1, 1].set_xlabel('Hours since last transaction')

# Gender
sns.countplot(data=fraud_df, x='gender', ax=axes[2, 0], palette='pastel')
axes[2, 0].set_title('Gender Distribution Among Clients')
axes[2, 0].set_xlabel('Gender')

# Transaction Count per hour of day
fraud_df['hour'] = pd.to_datetime(fraud_df['trans_date_trans_time']).dt.hour

sns.countplot(data=fraud_df, x='hour', ax=axes[2, 1], color='#FFB347')
axes[2, 1].set_title('Number of Transactions per Hour of the Day')
axes[2, 1].set_xlabel('Hour of Day (0–23)')
axes[2, 1].set_ylabel('Transaction Count')



plt.tight_layout()
plt.show()


### Relationships Between Variables and Target (Fraud vs Non-Fraud)

- **Amount:** Fraudulent transactions typically occur at higher amounts.  
- **Time of Day:** Fraud peaks late at night (10:00–03:00).  
- **Category:** `shopping_net`, `grocery_pos`, and `misc_net` show disproportionately high fraud activity.  
- **Age:** Slight skew toward older clients (30–40) and (50–70).  
- **Geography:** Fraud proportional to total transactions — no regional bias observed.


In [ ]:
category_order = sorted(for_training['category'].unique())

# Summarize both datasets
overall_counts = (
    for_training['category']
    .value_counts()
    .rename_axis('category')
    .reset_index(name='total_transactions')
)

fraud_counts = (
    fraud_df['category']
    .value_counts()
    .rename_axis('category')
    .reset_index(name='fraud_transactions')
)

# Merge to compare side by side
category_compare = pd.merge(overall_counts, fraud_counts, on='category', how='left').fillna(0)
category_compare['fraud_rate_%'] = (
    category_compare['fraud_transactions'] / category_compare['total_transactions']
) * 100
category_compare = category_compare.sort_values('total_transactions', ascending=False)

# Prepare job data
job_counts = for_training['job_sector'].value_counts().reset_index()
job_counts.columns = ['job_sector', 'total_transactions']

job_counts_fraud = fraud_df['job_sector'].value_counts().reset_index()
job_counts_fraud.columns = ['job_sector', 'fraud_transactions']

# --- Plot setup ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.set_style("whitegrid")

# All transactions by category
sns.barplot(
    data=category_compare,
    x='category', y='total_transactions',
    palette='pastel', ax=axes[0, 0], order=category_order
)
axes[0, 0].set_title('Total Transactions by Category (All Data)')
axes[0, 0].set_xlabel('Category')
axes[0, 0].set_ylabel('Total Transactions')
axes[0, 0].tick_params(axis='x', rotation=45)

# Fraudulent transactions by category
sns.barplot(
    data=category_compare.sort_values('fraud_transactions', ascending=False),
    x='category', y='fraud_transactions',
    palette='muted', ax=axes[0, 1], order=category_order
)
axes[0, 1].set_title('Fraudulent Transactions by Category (Fraud Only)')
axes[0, 1].set_xlabel('Category')
axes[0, 1].set_ylabel('Fraudulent Transactions')
axes[0, 1].tick_params(axis='x', rotation=45)

# Total transactions by job sector
sns.barplot(data=job_counts, x='job_sector', y='total_transactions', palette='pastel', ax=axes[1, 0])
axes[1, 0].set_title('Total Transactions by Job Sector')
axes[1, 0].set_xlabel('Job Sector')
axes[1, 0].set_ylabel('Total Transactions')
axes[1, 0].tick_params(axis='x', rotation=45)

# Fraudulent transactions by job sector
sns.barplot(data=job_counts_fraud, x='job_sector', y='fraud_transactions', palette='muted', ax=axes[1, 1])
axes[1, 1].set_title('Fraudulent Transactions by Job Sector')
axes[1, 1].set_xlabel('Job Sector')
axes[1, 1].set_ylabel('Fraudulent Transactions')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
job_fraud_summary = (
    for_training.groupby('job_sector')
    .agg(
        total_transactions=('is_fraud', 'count'),
        fraud_transactions=('is_fraud', 'sum')
    )
    .reset_index()
)

job_fraud_summary['fraud_rate_%'] = (
    job_fraud_summary['fraud_transactions'] / job_fraud_summary['total_transactions'] * 100
)
print(job_fraud_summary)

corr = job_fraud_summary['total_transactions'].corr(job_fraud_summary['fraud_transactions'])
print(f"\n\nCorrelation between total transactions and fraud counts: {corr:.2f}")


With a correlation coefficient of `1.00` between job-related transaction counts and fraud counts, the relationship is perfectly proportional.
This confirms that differences in fraud volume across job sectors are purely driven by transaction frequency rather than job type itself.
Therefore, job classification offers negligible predictive power for fraud detection in this dataset.

### Problem Type and Class Balance

This is a **binary classification** problem.  
Fraudulent transactions represent approximately **0.58%** of all transactions, indicating a **highly imbalanced dataset**.

Such imbalance can lead to biased models that favor the majority (non-fraudulent) class.  
In later stages, balancing techniques such as **oversampling (SMOTE)** or **class weighting** should be used to mitigate this issue.


--------------------------------------------------------------------

# Second Delivery

--------------------------------------------------------------------

In [ ]:
!pip install factor_analyzer
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity

In [ ]:
cols_to_keep = ['amt', 'city_pop', 'Age', 'is_fraud', 'time_since_last_trans','hour', 'avg_trans_amt','amt_ratio' ]
bank_numeric = for_training[cols_to_keep]
corr = bank_numeric.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    square=True
)
plt.title('Correlation Heatmap of Bank Features', fontsize=14)
plt.show()


###Correlation Analysis

The strongest correlation appears between amt (transaction amount) and amt_ratio (r ≈ 0.97). This indicates that these two variables contain almost the same information, since amt_ratio is derived from amt and the customer’s average transaction amount.

Moderate correlations are observed between amt and is_fraud (r ≈ 0.22) and between amt_ratio and is_fraud (r ≈ 0.20), suggesting that higher transaction values and unusually large relative amounts tend to be associated with fraudulent cases.

Other variables such as city_pop, Age, hour, and time_since_last_trans show very low correlations (|r| < 0.2), implying they contribute independent information.

To reduce multicollinearity, amt will be disregarded in the model, retaining amt_ratio as the more informative and normalized representation of transaction value. This helps prevent redundant predictors in subsequent analyses such as PCA and clustering.

In [ ]:
# Create new time variables
for_training['hour_sin'] = np.sin(2 * np.pi * for_training['hour'] / 24)
for_training['hour_cos'] = np.cos(2 * np.pi * for_training['hour'] / 24)
for_training['log_amt_ratio'] = np.log1p(for_training['amt_ratio'])
for_training['log_city_pop'] = np.log1p(for_training['city_pop'])


###Cyclical Encoding of Time-of-Day Variable and Standardizing Amount Ratio

The hour variable was transformed using sine and cosine encoding to preserve its cyclical nature (0–23). Linear representation would incorrectly treat 23:00 and 0:00 as distant points, although they occur consecutively.

The new features hour_sin and hour_cos allow the model to capture patterns of fraud concentrated during late-night and early-morning hours, improving the interpretability of temporal effects in both dimensionality reduction and feature selection.

The amt_ratio upon first attept to run the model demonstraded an extreme right-skewed behavior and negatively affected the PCA and clustering process. Therefore, we decided to standardize the variable and create a less drastic variable called log_am_ratio.

###Rule of thumb:

hour_sin > 0 → Morning to Noon

hour_sin < 0 → Afternoon to Night

Values near +1 = early morning; –1 = evening/night; 0 = midnight/noon transition.

In [ ]:
cols_to_keep = ['log_city_pop', 'Age', 'is_fraud', 'time_since_last_trans_hour', 'log_amt_ratio','hour_sin', 'hour_cos' ]
bank_numeric = for_training[cols_to_keep]
corr = bank_numeric.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    square=True
)
plt.title('Correlation Heatmap of Bank Features', fontsize=14)
plt.show()

In [ ]:
# Define all relevant numeric columns to test viability
test_cols = [
    'log_city_pop',
    'Age',
    'time_since_last_trans_hour',
    'avg_trans_amt',
    'log_amt_ratio',
    'hour_sin',
    'hour_cos'
]

In [ ]:
from factor_analyzer.factor_analyzer import calculate_kmo

numeric_df = for_training[test_cols]
sample_df = numeric_df.sample(10000, random_state=42)
chi_square_value, p_value = calculate_bartlett_sphericity(sample_df)
kmo_all, kmo_model = calculate_kmo(sample_df)

print(f'Chi Square Value: {chi_square_value}')
print(f'P_value: {p_value}')



In [ ]:
# KMO Test Summary

kmo_all, kmo_model = calculate_kmo(sample_df)

print("Overall KMO:", round(kmo_model, 3))

kmo_per_variable = pd.Series(kmo_all, index=sample_df.columns)
print(kmo_per_variable.sort_values())


In [ ]:
# LASSO Test Summary
X = for_training[test_cols].drop(columns=['is_fraud'], errors='ignore')
y = for_training['is_fraud']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_scaled, y)

# Get nonzero coefficients (selected features)
lasso_importance = pd.Series(lasso.coef_, index=X.columns)
lasso_importance = lasso_importance[lasso_importance != 0].sort_values(key=abs, ascending=False)

print("Selected features by Lasso:")
print(lasso_importance)



###Sampling Adequacy and Factorability Tests

Bartlett’s Test of Sphericity was significant (χ² = 1022.99, p < 0.001), indicating that the dataset’s correlation matrix is not an identity matrix — therefore, the data is suitable for dimensionality reduction.

The KMO measure of 0.51 suggests moderate sampling adequacy, meaning that while the data contains enough shared variance to justify PCA, some features may not be strongly interrelated.

Overall, these results support proceeding with PCA, though the components should be interpreted with caution.


###Variable Retention Justification

Both statistical (KMO) and predictive (LASSO) tests were used to assess feature relevance.

The KMO analysis indicated that **hour_cos** exhibited weak shared variance (KMO = 0.44) and was therefore removed to improve PCA suitability.

LASSO regression further confirmed that **hour_cos** contributed minimally to the predictive structure.
While **log_city_pop** was not predictive in LASSO, its acceptable KMO score (0.51) supports its retention for clustering, where structural variance is more important than predictive strength.

##Clustering

In [ ]:
# Here we define the columns we have decided to use for clustering
numeric_cols = [
    'log_city_pop',
    'Age',
    'time_since_last_trans_hour',
    'log_amt_ratio',
    'hour_sin'
]

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(for_training[numeric_cols])


pca = PCA( n_components = 4 ) # Set number of components here
pca_result = pca.fit_transform(scaled_data)

# Add PCA results to DataFrame
for_training['PCA1'] = pca_result[:, 0]
for_training['PCA2'] = pca_result[:, 1]

# Visualize PCA components colored by fraud
# --- Sample data ---
sample_df = for_training.sample(10000, random_state=42)

# --- Create figure with two side-by-side subplots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.set_style("whitegrid")

# --- Plot 1: Normal view (fraud + non-fraud) ---
sns.scatterplot(
    data=sample_df,
    x='PCA1', y='PCA2',
    hue='is_fraud',
    palette=['#4B9CD3', '#E15759'],
    alpha=0.6,
    ax=axes[0]
)
axes[0].set_title('PCA Projection: Fraud vs Non-Fraud (Full View)')
axes[0].legend(title='is_fraud', loc='upper right')

# --- Plot 2: Fraud highlighted only ---
sns.scatterplot(
    data=sample_df[sample_df['is_fraud'] == 0],
    x='PCA1', y='PCA2',
    color='lightgray',
    alpha=0.5,
    ax=axes[1],
    label='Non-Fraud (dimmed)'
)
sns.scatterplot(
    data=sample_df[sample_df['is_fraud'] == 1],
    x='PCA1', y='PCA2',
    color='#E15759',
    alpha=0.8,
    ax=axes[1],
    label='Fraud (highlighted)'
)
axes[1].set_title('PCA Projection: Fraud Highlighted')
axes[1].legend(title='Transaction Type', loc='upper right')

plt.tight_layout()
plt.show()





In [ ]:
# We confirm that 4 components are optimal
scaler = StandardScaler()
X_scaled = scaler.fit_transform(for_training[numeric_cols])


pca = PCA(n_components=5, random_state=42)
X_pca = pca.fit_transform(X_scaled)


explained_var = pca.explained_variance_ratio_
total_var = explained_var.sum() * 100

print("Explained variance by each component:", np.round(explained_var, 4))
print(f"Total variance explained: {total_var:.2f}%")


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, len(explained_var)+1), explained_var, marker='o', linestyle='--')
axes[0].set_title('Scree Plot: Explained Variance by Component')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].grid(True)

axes[1].plot(range(1, len(explained_var)+1),
             np.cumsum(explained_var),
             marker='o', linestyle='--', color='tab:blue')
axes[1].set_title('Cumulative Explained Variance by Component')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Cumulative Explained Variance Ratio')
axes[1].grid(True)

plt.tight_layout()
plt.show()

loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=numeric_cols
)

print("\nPCA Loadings (Feature Contributions to Each Component):")
display(loadings.round(3))


###PCA Analysis

The scree plot shows the proportion of variance explained by each principal component.
The first component (PC1) accounts for **26.19%** of total variance, followed by PC2 (**21.13%**),
PC3 (**20.19%**), and PC4 (**17.57%**). Together, the first four components explain approximately **85%**
of the total data variance.

The curve flattens after the fourth component, indicating diminishing returns from additional components.
This confirms that a **four-component PCA solution** is sufficient to summarize the dataset while retaining
most of its informational content. These components were subsequently used for clustering analysis.

In [ ]:
# Create a PCA biplot
def biplot(score, coeff, pcax=1, pcay=2, feature_labels=None, sample_labels=None):

    # Convert from 1-based to 0-based index
    pca1, pca2 = pcax - 1, pcay - 1

    xs = score[:, pca1]
    ys = score[:, pca2]

    # Scale scores to fit nicely within [-1,1]
    scalex = 1.0 / (xs.max() - xs.min())
    scaley = 1.0 / (ys.max() - ys.min())

    # Plot the sample scores
    plt.figure(figsize=(8, 6))
    plt.scatter(xs * scalex, ys * scaley, alpha=0.3, color='gray')

    # Optionally annotate samples
    if sample_labels is not None:
        for i, label in enumerate(sample_labels):
            plt.text(xs[i] * scalex, ys[i] * scaley, label,
                     color='blue', ha='center', va='center', fontsize=8)

    # Plot the feature loading vectors
    for i in range(coeff.shape[0]):
        plt.arrow(0, 0, coeff[i, pca1], coeff[i, pca2],
                  color='r', alpha=0.5, head_width=0.02)
        if feature_labels is not None:
            plt.text(coeff[i, pca1] * 1.15, coeff[i, pca2] * 1.15,
                     feature_labels[i], color='darkblue', ha='center', va='center', fontsize=9)

        else:
            plt.text(coeff[i, pca1] * 1.15, coeff[i, pca2] * 1.15,
                     f"Var{i+1}", color='darkblue', ha='center', va='center', fontsize=9)


    plt.xlim(-1, 1)
    plt.ylim(-1, 1)
    plt.xlabel(f"PC{pcax}")
    plt.ylabel(f"PC{pcay}")
    plt.title("PCA Biplot: Feature Contributions to Components")
    plt.grid(True)
    plt.show()


biplot(
    score=pca_result,
    coeff=pca.components_.T,
    pcax=1, pcay=2,
    feature_labels=numeric_cols
)


###Feature Selection Using LASSO Regression

LASSO regression identified the most relevant predictors of fraud.
The strongest variable is **log_amt_ratio**, indicating that transactions significantly larger than a customer’s usual amount are more likely to be fraudulent.

Time-based features also play an important role.
The positive effect of **hour_sin** shows that fraud tends to occur during late-night and early-morning hours, while the negative **time_since_last_trans_hour** suggests that fraudulent transactions often happen in quick succession.
Encoding hour cyclically allowed these daily patterns to be captured accurately.

In summary, fraud in this dataset is driven mainly by abnormal spending behavior and transaction timing, not by demographic characteristics.

##K-means Clustering ---------------------

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# --- Use PCA results ---
X_cluster = pca_result[:, :4]

# Random sample 20000 entries to lower computational load
np.random.seed(42)
sample_idx = np.random.choice(X_cluster.shape[0], size=20000, replace=False)
X_sample = X_cluster[sample_idx, :]

# Test different numbers of clusters
inertia = []
silhouette = []
K = range(2, 10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_sample)
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(X_sample, labels))

# Plot elbow and silhouette curves
plt.figure(figsize=(10,4))

# Elbow plot
plt.subplot(1,2,1)
plt.plot(K, inertia, 'bo--')
plt.title('Elbow Method (Inertia)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')

# Silhouette plot
plt.subplot(1,2,2)
plt.plot(K, silhouette, 'go--')
plt.title('Silhouette Score by Cluster Count')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')

plt.tight_layout()
plt.show()


###Optimal Cluster Selection

The number of clusters was determined using both the Elbow Method and the Silhouette Coefficient. The Elbow curve showed a clear inflection at k = 5, after which reductions in inertia diminished somewhat noticably. The Silhouette analysis also shows that the optimal range for k is between 5 and 7 clusters.
Since the silhouette peak 7 and the inertia elbow 5 are close, either value would be acceptable, but k = 5 offers simpler, more interpretable clusters..

Therefore, k = 5 was selected as the optimal number of clusters, balancing model interpretability and performance.

In [ ]:
final_kmeans = KMeans(n_clusters=5, random_state=42)
cluster_kmeans = final_kmeans.fit_predict(X_cluster)

for_training['cluster_km'] = cluster_kmeans
for_training.head(3)


In [ ]:
# Boxplot for K-means cluster analysis
features = ['amt','log_amt_ratio', 'time_since_last_trans_hour', 'Age', 'log_city_pop', 'hour_sin']

plt.figure(figsize=(15,10))
for i, col in enumerate(features, 1):
    plt.subplot(3, 2, i)
    sns.boxplot(data=for_training, x='cluster_km', y=col, palette='Set2')
    plt.title(f'{col} by Cluster')
plt.tight_layout()
plt.show()

| hour_sin | Approx. Time     | Period                      |
|-----------|------------------|------------------------------|
| +1.0      | 06:00            | Early Morning               |
| +0.7      | 08:00            | Morning                     |
| +0.5      | 04:00–05:00      | Pre-Dawn                    |
| +0.3      | 02:00            | Late Night / Early Morning  |
|  0.0      | 00:00 / 12:00    | Midnight or Noon            |
| –0.3      | 14:00            | Early Afternoon             |
| –0.5      | 16:00            | Afternoon                   |
| –0.7      | 18:00            | Evening                     |
| –0.9      | 20:00–22:00      | Late Evening                |
| –1.0      | 18:00            | Sunset / Evening Peak       |


### K-means Clustering Box Plot Interpretation

K-Means clustering behavioral distinctions:

- **Cluster 1** shows the highest transaction amounts and `log_amt_ratio`, representing high-value, irregular spenders — a potential high-risk group.
- **Cluster 3** has longer gaps between transactions and older customers, suggesting infrequent, irregular activity.
- **Clusters 0, 2, and 4** show lower transaction amounts and more regular patterns; Cluster 4, however, is associated with nighttime activity (`hour_sin` < 0).
- **City population** varies by cluster, with Clusters 0 and 3 concentrated in large cities.

Overall, clusters can be interpreted as distinct customer segments varying by transaction magnitude, timing, and frequency. Fraud appears more concentrated among high-value or irregular clusters (1 and 3), indicating that transaction behavior and time patterns play a key role in differentiating fraud risk.

In [ ]:
X = for_training[numeric_cols]
y = for_training['cluster_km']

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.plot(kind='bar', title='Feature Importance for Cluster Membership', figsize=(8,4))
plt.show()

importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances)


In [ ]:
# This code shows us the fraud rate per cluster
fraud_rates = for_training.groupby('cluster_km')['is_fraud'].mean().sort_values()

plt.figure(figsize=(7,5))
sns.barplot(x=fraud_rates.index, y=fraud_rates.values, palette='Reds')
plt.title('Fraud Rate by Cluster')
plt.xlabel('Cluster')
plt.ylabel('Fraud Rate')
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_cluster[:, 0],
    y=X_cluster[:, 1],
    hue=for_training['cluster_km'],
    palette='Set2',
    alpha=0.6
)
plt.title('K-Means Clusters Visualized on PCA Components')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend(title='Cluster')
plt.show()

In [ ]:
unique_clusters = np.unique(for_training['cluster_km'])

plt.figure(figsize=(14, 10))

for i, c in enumerate(unique_clusters):
    plt.subplot(3, 2, i+1)

    # dim all points
    sns.scatterplot(
        x=X_cluster[:, 0],
        y=X_cluster[:, 1],
        hue=for_training['cluster_km'],
        palette='Set2',
        alpha=0.3,
        legend=False
    )

    # highlight the current cluster
    mask = for_training['cluster_km'] == c
    sns.scatterplot(
        x=X_cluster[mask, 0],
        y=X_cluster[mask, 1],
        color=sns.color_palette('Set2')[int(c)],
        alpha=0.9,
        s=40,
        label=f'Cluster {c}'
    )

    plt.title(f'Cluster {c} Highlighted')
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')

plt.tight_layout()
plt.show()


In [ ]:


plt.figure(figsize=(14, 10))

for i, c in enumerate(unique_clusters):
    plt.subplot(3, 2, i+1)

    # dim all points
    sns.scatterplot(
        x=X_cluster[:, 0],
        y=X_cluster[:, 1],
        hue=for_training['cluster_km'],
        palette='Set2',
        alpha=0.2,
        legend=False
    )

    # highlight the current cluster
    mask = for_training['cluster_km'] == c
    sns.scatterplot(
        x=X_cluster[mask, 0],
        y=X_cluster[mask, 1],
        color=sns.color_palette('Set2')[int(c)],
        alpha=0.6,
        s=40,
        label=f'Cluster {c}'
    )

    mask_fraud = (for_training['cluster_km'] == c) & (for_training['is_fraud'] == 1)
    sns.scatterplot(
        x=X_cluster[mask_fraud, 0],
        y=X_cluster[mask_fraud, 1],
        color='red',
        s=60,
        alpha = 0.9,
        label='Fraudulent Transactions'
    )

    plt.title(f'Cluster {c} Highlighted')
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')

plt.tight_layout()
plt.show()







###Summary per Cluster

In [ ]:
summary_df_k = (
    for_training.groupby('cluster_km')
      .agg(
          n=('is_fraud', 'size'),
          fraud_rate=('is_fraud', 'mean'),
          log_amt_ratio_median=('log_amt_ratio', 'median'),
          time_since_last_trans_hour_median=('time_since_last_trans_hour', 'median'),
          hour_sin_median=('hour_sin', 'median'),
          Age_median=('Age', 'median'),
          log_city_pop_median=('log_city_pop', 'median')
      )
      .round(3)
)
summary_df_k

##Agglomerative Clustering  ---------------------

In [ ]:
# Here we determine 4 as the optimal number of clusters
from sklearn.cluster import AgglomerativeClustering

X_cluster = pca_result[:, :4]

# Random sample 10000 entries to lessen computational load
np.random.seed(42)
sample_idx = np.random.choice(X_cluster.shape[0], size=10000, replace=False)
X_sample = X_cluster[sample_idx, :]

# Test different numbers of clusters
K = range(2, 10)
silhouette_scores = []
wcss_values = []

for k in K:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = agg.fit_predict(X_sample)

    # Silhouette Score
    sil = silhouette_score(X_sample, labels)
    silhouette_scores.append(sil)

    # Approximate Within-Cluster Sum of Squares (WCSS)
    wcss = 0
    for cluster_id in np.unique(labels):
        cluster_points = X_sample[labels == cluster_id]
        centroid = cluster_points.mean(axis=0)
        wcss += ((cluster_points - centroid) ** 2).sum()
    wcss_values.append(wcss)

# Plot both curves side by side
plt.figure(figsize=(12, 5))

# Elbow plot
plt.subplot(1, 2, 1)
plt.plot(K, wcss_values, 'bo--', linewidth=2, markersize=6)
plt.title('Elbow Method (Approx. WCSS)', fontsize=12)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares')

# Silhouette plot
plt.subplot(1, 2, 2)
plt.plot(K, silhouette_scores, 'go--', linewidth=2, markersize=6)
plt.title('Silhouette Score by Cluster Count', fontsize=12)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Average Silhouette Score')

plt.tight_layout()
plt.show()


###Optimal Cluster Selection

Like with the K-means clustering, the number of clusters was determined using both the Elbow Method and the Silhouette Coefficient. The Elbow curve showed a clear inflection at k = 4, after which reductions in inertia diminished somewhat noticeably. The Silhouette analysis also shows that the optimal range for k at 4 and 7 clusters.
Since we believe that 7 clusters would be rather difficult to handle we decided to settle for 4 clusters, for easier interpretation.

In [ ]:
from sklearn.metrics import silhouette_score, adjusted_rand_score

sample_size = 20000
df_sample_agg = for_training.sample(n=sample_size, random_state=42)

features = numeric_cols

# Scale and transform only the sample
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_sample_agg[features])

pca = PCA(n_components=4, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Run Agglomerative clustering on the sample
agg = AgglomerativeClustering(
    n_clusters=4,
    linkage='ward',
    compute_distances=True
)
agg_labels = agg.fit_predict(X_pca)

# Attach to sample DataFrame
df_sample_agg['agg_cluster'] = agg_labels

# Evaluate and interpret on the sample
sil_agg = silhouette_score(X_pca, agg_labels)
print(f"Silhouette (Agglomerative, k=4) [sampled {sample_size:,} rows]: {sil_agg:.3f}")


In [ ]:
# Boxplot for Agglomerative cluster analysis
features = ['log_amt_ratio', 'time_since_last_trans_hour', 'Age', 'log_city_pop', 'hour_sin']

plt.figure(figsize=(15, 10))

for i, col in enumerate(features, 1):
    plt.subplot(3, 2, i)
    sns.boxplot(
        data=df_sample_agg,
        x='agg_cluster',
        y=col,
        palette='Set2'
    )
    plt.title(f'{col} by Agglomerative Cluster', fontsize=11)
    plt.xlabel('Cluster')
    plt.ylabel(col)

plt.tight_layout()
plt.show()

| hour_sin | Approx. Time     | Period                      |
|-----------|------------------|------------------------------|
| +1.0      | 06:00            | Early Morning               |
| +0.7      | 08:00            | Morning                     |
| +0.5      | 04:00–05:00      | Pre-Dawn                    |
| +0.3      | 02:00            | Late Night / Early Morning  |
|  0.0      | 00:00 / 12:00    | Midnight or Noon            |
| –0.3      | 14:00            | Early Afternoon             |
| –0.5      | 16:00            | Afternoon                   |
| –0.7      | 18:00            | Evening                     |
| –0.9      | 20:00–22:00      | Late Evening                |
| –1.0      | 18:00            | Sunset / Evening Peak       |


### Agglomerative Clustering Box Plot Analysis

Agglomerative clustering behavioral distinctions:

- **Cluster 0:** Users from smaller cities with moderately higher transaction ratios and nighttime spending patterns.  
- **Cluster 1:** Older, low-value, daytime spenders showing consistent and regular behavior.  
- **Cluster 2:** Older customers with long intervals between transactions, indicating infrequent or irregular usage.  
- **Cluster 3:** Younger, urban users active during nighttime, representing digitally engaged but potentially higher-risk profiles.

Overall, Agglomerative clustering reinforces the behavioral structure found with K-Means.  
Transaction frequency (`time_since_last_trans_hour`) and timing (`hour_sin`) continue to be strong discriminators of cluster membership.  
Clusters 0 and 3 show patterns (off-hour and urban activity) commonly associated with elevated fraud risk.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df_sample_agg[['log_amt_ratio', 'time_since_last_trans_hour', 'Age',
               'log_city_pop', 'hour_sin']]
y = df_sample_agg['agg_cluster']

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X, y)

# --- Plot importance ---
plt.figure(figsize=(8,4))
importances.plot(kind='bar', color='skyblue')
plt.title('Feature Importance for Agglomerative Cluster Membership', fontsize=12)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances)


In [ ]:
fraud_rates = (
    df_sample_agg.groupby('agg_cluster')['is_fraud']
    .mean()
)

# Plot
plt.figure(figsize=(7,5))
sns.barplot(
    x=fraud_rates.index.astype(str),   # cluster labels as strings for clean x-axis
    y=fraud_rates.values,
    palette='Reds'
)
plt.title('Fraud Rate by Agglomerative Cluster', fontsize=13)
plt.xlabel('Agglomerative Cluster')
plt.ylabel('Average Fraud Rate')
plt.tight_layout()
plt.show()



In [ ]:
# Scatter on first two PCs
pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Cluster': agg_labels   # or for_training['cluster'] if that’s your variable
})

# Seaborn scatterplot
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=pca_df,
    x='PC1',
    y='PC2',
    hue='Cluster',
    palette='Set2',
    alpha=0.5,
    s=20,        # adjust point size if desired
    edgecolor='white'
)
plt.title('Agglomerative Clusters Visualized on PCA Components')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend(title='Cluster', loc='best')
plt.show()


In [ ]:
X_cluster = X_pca
cluster_labels = df_sample_agg['agg_cluster'].values
unique_clusters = np.unique(cluster_labels)

plt.figure(figsize=(14, 10))

# Loop through clusters and highlight each one
for i, c in enumerate(unique_clusters):
    plt.subplot(3, 2, i+1)

    # Dim all points
    sns.scatterplot(
        x=X_cluster[:, 0],
        y=X_cluster[:, 1],
        hue=cluster_labels,
        palette='Set2',
        alpha=0.2,
        legend=False
    )

    # Highlight the current cluster
    mask = cluster_labels == c
    sns.scatterplot(
        x=X_cluster[mask, 0],
        y=X_cluster[mask, 1],
        color=sns.color_palette('Set2')[int(c)],
        alpha=0.9,
        s=40,
        edgecolor='w',
        linewidth=0.3,
        label=f'Cluster {c}'
    )

    plt.title(f'Agglomerative Cluster {c} Highlighted', fontsize=12)
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')

plt.tight_layout()
plt.show()


In [ ]:
X_cluster = X_pca
cluster_labels = df_sample_agg['agg_cluster'].values
fraud_labels = df_sample_agg['is_fraud'].values
unique_clusters = np.unique(cluster_labels)

plt.figure(figsize=(14, 10))

for i, c in enumerate(unique_clusters):
    plt.subplot(3, 2, i+1)

    sns.scatterplot(
        x=X_cluster[:, 0],
        y=X_cluster[:, 1],
        hue=cluster_labels,
        palette='Set2',
        alpha=0.15,
        legend=False
    )

    mask = cluster_labels == c
    sns.scatterplot(
        x=X_cluster[mask, 0],
        y=X_cluster[mask, 1],
        color=sns.color_palette('Set2')[int(c)],
        alpha=0.8,
        s=35,
        edgecolor='k',
        linewidth=0.2,
        label=f'Cluster {c}'
    )

    mask_fraud = (cluster_labels == c) & (fraud_labels == 1)
    sns.scatterplot(
        x=X_cluster[mask_fraud, 0],
        y=X_cluster[mask_fraud, 1],
        color='red',
        s=50,
        alpha=0.9,
        label='Fraudulent Txns'
    )

    plt.title(f'Cluster {c} Highlighted')
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')

plt.tight_layout()
plt.show()


In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)

print(importances.sort_values(ascending=False).round(3))


In [ ]:
summary_agg = (
    df_sample_agg.groupby('agg_cluster')
    .agg({
        'is_fraud': ['mean', 'sum', 'count'],
        'log_amt_ratio': 'mean',
        'time_since_last_trans_hour': 'mean',
        'Age': 'mean',
        'log_city_pop': 'mean',
        'hour_sin': 'mean'
    })
    .round(2)
)

summary_agg.columns = [
    'fraud_rate', 'fraud_count', 'cluster_size',
    'avg_log_amt_ratio', 'avg_time_since_last_hour',
    'avg_age', 'avg_log_city_pop', 'avg_hour_sin'
]

summary_agg = summary_agg.sort_values('fraud_rate', ascending=False)

summary_agg



--------------------------------------------------------------------

#Third Delivery

--------------------------------------------------------------------

In [ ]:
for_training.head()

In [ ]:
for_training.info()

##Data Preparation for Training
Here we begin preparing the data for our machine learning models by separating the target variable from the rest of the dataset. We set y as the target variable (is_fraud), which indicates whether each transaction is fraudulent or not. For X, we keep only the features we want the model to learn from. This means we drop the target column itself, along with any other variables that do not add useful information or could cause issues during training (such as IDs, timestamps, or other non-predictive fields).

This step ensures that the model focuses only on the relevant input features, avoids data leakage, and keeps the training process clean and consistent. Once X and y are defined properly, we can safely proceed with the train/test split and the sampling strategies.

In [ ]:
# Target variable
y = for_training['is_fraud']

# Feature variables
X = for_training.drop(['is_fraud', 'trans_date_trans_time','time_since_last_trans', 'cc_num'], axis=1)


In [ ]:
cat_cols = ['category', 'gender', 'city', 'state', 'job', 'job_sector', 'amt_bin']


In [ ]:
# Columns to Label Encode (high cardinality)
label_cols = ['city', 'state', 'job']
le = LabelEncoder()
for col in label_cols:
    X[col] = le.fit_transform(X[col].astype(str))

# Columns to One-Hot Encode (small cardinality)
ohe_cols = ['gender', 'amt_bin', 'job_sector', 'category']
X = pd.get_dummies(X, columns=ohe_cols, drop_first=True)


In [ ]:
def clean_col(name):
    return (
        name.replace('[','(')
            .replace(']',')')
            .replace('<','lt')
            .replace(' ', '_')
    )

X.columns = [clean_col(c) for c in X.columns]


In [ ]:
# Take a 20% representative sample of the full dataset
X_small, _, y_small, _ = train_test_split(
    X, y, train_size=0.20, stratify=y, random_state=42
)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_small, y_small, test_size=0.2, stratify=y_small, random_state=42
)


In [ ]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [ ]:
model_name = ''

In [ ]:
X_train.info()

# Oversampling

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_over, y_train_over = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_over.value_counts())


In [ ]:
print(y_test.unique())
print(y_test.value_counts())


##Logistic Regression (Oversampling)

In [ ]:
model_name = 'Logistic Regression (Oversampling)'


model = LogisticRegression(max_iter=500, n_jobs=-1)
model.fit(X_train_over, y_train_over)

# 1) Predictions
y_pred_lr = model.predict(X_test)
y_proba_lr = model.predict_proba(X_test)[:, 1]

threshold = 0.4
y_pred_lr_thresh = (y_proba_lr >= threshold).astype(int)

# 2) Confusion Matrix
print("Confusion Matrix (Logistic Regression – Oversampling):")
print(confusion_matrix(y_test, y_pred_lr_thresh))

# 3) Classification Report
print("\nClassification Report (Logistic Regression – Oversampling):")
print(classification_report(y_test, y_pred_lr_thresh, digits=4))

# 4) Extra Metrics (same formatting)
precision_lr = precision_score(y_test, y_pred_lr_thresh)
recall_lr = recall_score(y_test, y_pred_lr_thresh)
f1_lr = f1_score(y_test, y_pred_lr_thresh)
roc_auc_lr = roc_auc_score(y_test, y_proba_lr)

print(f"Precision (LR Oversampling): {precision_lr:.4f}")
print(f"Recall    (LR Oversampling): {recall_lr:.4f}")
print(f"F1-score  (LR Oversampling): {f1_lr:.4f}")
print(f"ROC-AUC   (LR Oversampling): {roc_auc_lr:.4f}")

##Random Forest Classifier (Oversampling)

In [ ]:
model_name = 'Random Forest Classifier (Oversampling)'

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42
)
model.fit(X_train_over, y_train_over)

# 1) Predictions
y_pred_rf = model.predict(X_test)
y_proba_rf = model.predict_proba(X_test)[:, 1]   # Prob. of fraud

threshold = 0.4
y_pred_over_thresh = (y_proba_rf >= threshold).astype(int)

# 2) Confusion Matrix
print("Confusion Matrix (RF Oversampling):")
print(confusion_matrix(y_test, y_pred_rf))

# 3) Classification Report
print("\nClassification Report (RF Oversampling):")
print(classification_report(y_test, y_pred_rf, digits=4))

# 4) Extra Metrics (matching undersampling format)
precision_over = precision_score(y_test, y_pred_rf)
recall_over = recall_score(y_test, y_pred_rf)
f1_over = f1_score(y_test, y_pred_rf)
roc_auc_over = roc_auc_score(y_test, y_proba_rf)

print(f"Precision (RF Oversampling): {precision_over:.4f}")
print(f"Recall (RF Oversampling):    {recall_over:.4f}")
print(f"F1-score (RF Oversampling):  {f1_over:.4f}")
print(f"ROC-AUC (RF Oversampling):   {roc_auc_over:.4f}")





##XGBoost (Oversampling)

In [ ]:
model_name = 'XGBoost (Oversampling)'


# Train XGBoost model
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

model.fit(X_train_over, y_train_over)

# 1) Predictions
y_pred_xgb = model.predict(X_test)
y_proba_xgb = model.predict_proba(X_test)[:, 1]

# Optional threshold (same as your previous models)
threshold = 0.4
y_pred_xgb_thresh = (y_proba_xgb >= threshold).astype(int)

# 2) Confusion Matrix
print("Confusion Matrix (XGBoost – Oversampling):")
print(confusion_matrix(y_test, y_pred_xgb_thresh))

# 3) Classification Report
print("\nClassification Report (XGBoost – Oversampling):")
print(classification_report(y_test, y_pred_xgb_thresh, digits=4))

# 4) Extra Metrics
precision_xgb = precision_score(y_test, y_pred_xgb_thresh)
recall_xgb = recall_score(y_test, y_pred_xgb_thresh)
f1_xgb = f1_score(y_test, y_pred_xgb_thresh)
roc_auc_xgb = roc_auc_score(y_test, y_proba_xgb)

print(f"Precision (XGB Oversampling): {precision_xgb:.4f}")
print(f"Recall    (XGB Oversampling): {recall_xgb:.4f}")
print(f"F1-score  (XGB Oversampling): {f1_xgb:.4f}")
print(f"ROC-AUC   (XGB Oversampling): {roc_auc_xgb:.4f}")

##Apply random search to chosen model (XGBoost)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score

# Base model
xgb_base = XGBClassifier(
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

# Search space
param_dist = {
    'n_estimators': [300, 400, 500, 600, 800],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.07],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 1, 5, 10],
    'min_child_weight': [1, 3, 5, 7]
}

# Optimize for F1-score (fraud detection friendly)
scorer = make_scorer(f1_score)

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=25,
    scoring=scorer,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Fit search on OVERSAMPLED training data
random_search.fit(X_train_over, y_train_over)

# Print best parameters
print("Best Parameters:")
print(random_search.best_params_)


##XGBoost Tuned (Oversampling)

In [ ]:
model_name = 'XGBoost Improved (Oversampling)'

best_params = random_search.best_params_

model = XGBClassifier(
    **best_params,
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

model.fit(X_train_over, y_train_over)

# 1) Predictions
y_pred_xgb = model.predict(X_test)
y_proba_xgb = model.predict_proba(X_test)[:, 1]

# Optional threshold (same as your previous models)
threshold = 0.4
y_pred_xgb_thresh = (y_proba_xgb >= threshold).astype(int)

# 2) Confusion Matrix
print("Confusion Matrix (XGBoost – Oversampling):")
print(confusion_matrix(y_test, y_pred_xgb_thresh))

# 3) Classification Report
print("\nClassification Report (XGBoost – Oversampling):")
print(classification_report(y_test, y_pred_xgb_thresh, digits=4))

# 4) Extra Metrics
precision_xgb = precision_score(y_test, y_pred_xgb_thresh)
recall_xgb = recall_score(y_test, y_pred_xgb_thresh)
f1_xgb = f1_score(y_test, y_pred_xgb_thresh)
roc_auc_xgb = roc_auc_score(y_test, y_proba_xgb)

print(f"Precision (XGB Oversampling): {precision_xgb:.4f}")
print(f"Recall    (XGB Oversampling): {recall_xgb:.4f}")
print(f"F1-score  (XGB Oversampling): {f1_xgb:.4f}")
print(f"ROC-AUC   (XGB Oversampling): {roc_auc_xgb:.4f}")


# Undersampling

Even if we have determined that XGBoost using oversampling is sufficient we wanted to test our initial hypothesis and see if it in fact performs better than underfitting in this scenario

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)

X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("Before Undersampling:", y_train.value_counts())
print("After Undersampling:", y_train_under.value_counts())

##Random Forest (Undersampling)

In [ ]:
model_name = 'Random Forest Classifier (Undersampling)'

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42
)

model.fit(X_train_under, y_train_under)

# 1) Predictions
y_pred_under = model.predict(X_test)
y_proba_under = model.predict_proba(X_test)[:, 1]

# 2) Confusion matrix
print("Confusion Matrix (Undersampling):")
print(confusion_matrix(y_test, y_pred_under))

# 3) Main metrics
print("\nClassification Report (Undersampling):")
print(classification_report(y_test, y_pred_under, digits=4))

# 4) Extra key metrics
precision = precision_score(y_test, y_pred_under)
recall = recall_score(y_test, y_pred_under)
f1 = f1_score(y_test, y_pred_under)
roc_auc = roc_auc_score(y_test, y_proba_under)

print(f"Precision (Undersampling): {precision:.4f}")
print(f"Recall (Undersampling):    {recall:.4f}")
print(f"F1-score (Undersampling):  {f1:.4f}")
print(f"ROC-AUC (Undersampling):   {roc_auc:.4f}")


##XGBoost Classifier (Undersampling)

In [ ]:
model_name = 'XGBoost Classifier (Undersampling)'


best_params = random_search.best_params_

model = XGBClassifier(
    **best_params,
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

model.fit(X_train_under, y_train_under)

# 1) Predictions
y_pred_xgb = model.predict(X_test)
y_proba_xgb = model.predict_proba(X_test)[:, 1]

# Optional threshold (same as your previous models)
threshold = 0.4
y_pred_xgb_thresh = (y_proba_xgb >= threshold).astype(int)

# 2) Confusion Matrix
print("Confusion Matrix (XGBoost – Undersampling):")
print(confusion_matrix(y_test, y_pred_xgb_thresh))

# 3) Classification Report
print("\nClassification Report (XGBoost – Undersampling):")
print(classification_report(y_test, y_pred_xgb_thresh, digits=4))

# 4) Extra Metrics
precision_xgb = precision_score(y_test, y_pred_xgb_thresh)
recall_xgb = recall_score(y_test, y_pred_xgb_thresh)
f1_xgb = f1_score(y_test, y_pred_xgb_thresh)
roc_auc_xgb = roc_auc_score(y_test, y_proba_xgb)

print(f"Precision (XGB Undersampling): {precision_xgb:.4f}")
print(f"Recall    (XGB Undersampling): {recall_xgb:.4f}")
print(f"F1-score  (XGB Undersampling): {f1_xgb:.4f}")
print(f"ROC-AUC   (XGB Undersampling): {roc_auc_xgb:.4f}")

--------------------------------------------------------------------

#Compare Training Performance

--------------------------------------------------------------------

In [ ]:
print("====== TRAINING PERFORMANCE ======\n")
print(f'Model Being Tested: {model_name}')
print(f'Internal model class: {type(model).__name__}\n\n')

# ------------------------------
# TRAINING PERFORMANCE
# ------------------------------

# Predictions on training data
y_pred_proba_train = model.predict_proba(X_train_over)[:, 1]
train_threshold = 0.4
y_pred_train = (y_pred_proba_train >= train_threshold).astype(int)

print("Confusion Matrix (TRAINING):")
print(confusion_matrix(y_train_over, y_pred_train))

print("\nClassification Report (TRAINING):")
print(classification_report(y_train_over, y_pred_train))

print(f"Precision (Train): {precision_score(y_train_over, y_pred_train):.4f}")
print(f"Recall    (Train): {recall_score(y_train_over, y_pred_train):.4f}")
print(f"F1-score  (Train): {f1_score(y_train_over, y_pred_train):.4f}")
print(f"ROC-AUC   (Train): {roc_auc_score(y_train_over, y_pred_proba_train):.4f}")


print("\n\n====== TESTING PERFORMANCE ======")

# ------------------------------
# TEST PERFORMANCE
# ------------------------------

y_pred_proba_test = model.predict_proba(X_test)[:, 1]
test_threshold = 0.4
y_pred_test = (y_pred_proba_test >= test_threshold).astype(int)

print("Confusion Matrix (TEST):")
print(confusion_matrix(y_test, y_pred_test))

print("\nClassification Report (TEST):")
print(classification_report(y_test, y_pred_test))

print(f"Precision (Test): {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall    (Test): {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-score  (Test): {f1_score(y_test, y_pred_test):.4f}")
print(f"ROC-AUC   (Test): {roc_auc_score(y_test, y_pred_proba_test):.4f}")
